# Hybrid Movie Recommender System

A two-layer recommender that combines:
- **Collaborative Filtering** (SVD matrix factorization) — learns user preferences from rating history
- **Semantic Search** (sentence-transformers + FAISS) — finds movies with similar content/vibe
- **Hybrid fusion** — blends both scores for better cold-start handling and richer recommendations

Dataset: [MovieLens 100K](https://grouplens.org/datasets/movielens/100k/)

In [ ]:
%pip install scikit-surprise sentence-transformers faiss-cpu pandas numpy requests tqdm --quiet

In [1]:
import pandas as pd
import numpy as np
import requests
import zipfile
import os
from io import BytesIO
from pathlib import Path

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

from sentence_transformers import SentenceTransformer
import faiss

DATA_DIR = Path("data")

/Users/nikitaarora/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/nikitaarora/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Data Loading — MovieLens 100K

In [2]:
ML100K_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"

def download_movielens():
    if (DATA_DIR / "ml-100k").exists():
        print("Already downloaded.")
        return
    print("Downloading MovieLens 100K...")
    r = requests.get(ML100K_URL)
    with zipfile.ZipFile(BytesIO(r.content)) as z:
        z.extractall(DATA_DIR)
    print("Done.")

download_movielens()

# Ratings
ratings = pd.read_csv(
    DATA_DIR / "ml-100k" / "u.data",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"]
)

# Movies with genre flags (19 binary genre columns)
genre_cols = [
    "unknown", "Action", "Adventure", "Animation", "Children",
    "Comedy", "Crime", "Documentary", "Drama", "Fantasy",
    "Film-Noir", "Horror", "Musical", "Mystery", "Romance",
    "Sci-Fi", "Thriller", "War", "Western"
]
movies = pd.read_csv(
    DATA_DIR / "ml-100k" / "u.item",
    sep="|",
    encoding="latin-1",
    names=["item_id", "title", "release_date", "video_release_date", "imdb_url"] + genre_cols
)

# Build a text description for each movie: "Title | Genre1 Genre2 ..."
movies["genres_text"] = movies[genre_cols].apply(
    lambda row: " ".join(g for g, v in zip(genre_cols, row) if v == 1), axis=1
)
movies["description"] = movies["title"] + " | " + movies["genres_text"]

print(f"Ratings: {len(ratings):,}  |  Movies: {len(movies):,}  |  Users: {ratings['user_id'].nunique():,}")
movies[["item_id", "title", "description"]].head(5)

Already downloaded.
Ratings: 100,000  |  Movies: 1,682  |  Users: 943


,item_id,title,description
0,1,Toy Story (1995),Toy Story (1995) | Animation Children Comedy
1,2,GoldenEye (1995),GoldenEye (1995) | Action Adventure Thriller
2,3,Four Rooms (1995),Four Rooms (1995) | Thriller
3,4,Get Shorty (1995),Get Shorty (1995) | Action Comedy Drama
4,5,Copycat (1995),Copycat (1995) | Crime Drama Thriller


## 2. Collaborative Filtering — SVD Matrix Factorization

SVD decomposes the user-item rating matrix into latent factor vectors.  
Given user `u` and item `i`, the predicted rating is:

```
r̂(u,i) = μ + b_u + b_i + qᵢᵀ · pᵤ
```

where `p`, `q` are learned user/item embeddings and `b` are bias terms.

In [3]:
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings[["user_id", "item_id", "rating"]], reader)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

svd = SVD(n_factors=100, n_epochs=30, lr_all=0.005, reg_all=0.02, random_state=42)
svd.fit(trainset)

predictions = svd.test(testset)
rmse = accuracy.rmse(predictions)
print(f"SVD RMSE on test set: {rmse:.4f}")

RMSE: 0.9434
SVD RMSE on test set: 0.9434


In [4]:
def cf_scores_for_user(user_id: int, movie_ids: list[int]) -> dict[int, float]:
    """Return predicted CF rating for each movie_id (0-1 normalized)."""
    raw = {mid: svd.predict(user_id, mid).est for mid in movie_ids}
    min_r, max_r = min(raw.values()), max(raw.values())
    span = max_r - min_r or 1
    return {mid: (r - min_r) / span for mid, r in raw.items()}

# Sanity check
sample = cf_scores_for_user(1, [1, 2, 3, 50, 100])
print("CF scores (normalized) for user 1:", sample)

CF scores (normalized) for user 1: {1: 0.6607663417664, 2: 0.02464552941149158, 3: 0.0, 50: 1.0, 100: 0.7284457423632386}


## 3. Semantic Layer — Sentence Embeddings + FAISS

Each movie is embedded as a dense vector from its title + genres description.  
At query time, a natural-language prompt is embedded in the same space and we retrieve nearest neighbours via FAISS.

In [5]:
EMBED_MODEL = "all-MiniLM-L6-v2"  # 384-dim, fast, good quality

encoder = SentenceTransformer(EMBED_MODEL)

print(f"Encoding {len(movies)} movies...")
descriptions = movies["description"].tolist()
embeddings = encoder.encode(descriptions, batch_size=128, show_progress_bar=True, normalize_embeddings=True)

# item_id → row index mapping
item_id_to_idx = {row.item_id: i for i, row in enumerate(movies.itertuples())}
idx_to_item_id = {i: row.item_id for i, row in enumerate(movies.itertuples())}

# FAISS index (inner product on L2-normalized vectors == cosine similarity)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype(np.float32))

print(f"FAISS index built. Vectors: {index.ntotal}, Dim: {dim}")

Encoding 1682 movies...


Batches: 100%|██████████| 14/14 [00:00<00:00, 21.22it/s]

FAISS index built. Vectors: 1682, Dim: 384


In [6]:
def semantic_scores(query: str, movie_ids: list[int], top_k: int = 200) -> dict[int, float]:
    """Return cosine similarity scores (0-1) for movies matching a text query."""
    q_vec = encoder.encode([query], normalize_embeddings=True).astype(np.float32)
    k = min(top_k, index.ntotal)
    sims, idxs = index.search(q_vec, k)
    # sims are already in [-1,1] since vectors are normalized; shift to [0,1]
    scores = {idx_to_item_id[idx]: float((sim + 1) / 2)
              for idx, sim in zip(idxs[0], sims[0])
              if idx_to_item_id[idx] in movie_ids}
    return scores

# Sanity check
test_scores = semantic_scores("space adventure sci-fi", list(movies["item_id"])[:500])
top5 = sorted(test_scores.items(), key=lambda x: -x[1])[:5]
for mid, score in top5:
    title = movies.loc[movies.item_id == mid, "title"].values[0]
    print(f"  {title}: {score:.3f}")

  Star Trek IV: The Voyage Home (1986): 0.823
  Star Trek: Generations (1994): 0.819
  Star Trek: The Motion Picture (1979): 0.794
  Star Wars (1977): 0.793
  Star Trek V: The Final Frontier (1989): 0.792


## 4. Hybrid Recommender

Final score is a weighted blend:

```
score(u, i) = α · CF_score(u, i) + (1 - α) · Semantic_score(query, i)
```

- `α = 1.0` → pure collaborative filtering  
- `α = 0.0` → pure semantic search  
- `α = 0.5` → equal blend (default)  

Movies the user has already rated are excluded from results.

In [7]:
# Pre-compute each user's rated movies for fast lookup
user_rated = ratings.groupby("user_id")["item_id"].apply(set).to_dict()
all_movie_ids = set(movies["item_id"].tolist())


def hybrid_recommend(
    user_id: int,
    query: str = "",
    n: int = 10,
    alpha: float = 0.5
) -> pd.DataFrame:
    """
    Returns top-n movie recommendations for a user.

    Parameters
    ----------
    user_id : int
    query   : str   Natural-language mood/genre hint (e.g. "funny romantic comedy")
    n       : int   Number of results
    alpha   : float Weight for CF score (1-alpha goes to semantic). Ignored if query is empty.
    """
    rated = user_rated.get(user_id, set())
    candidates = list(all_movie_ids - rated)

    cf = cf_scores_for_user(user_id, candidates)

    if query.strip():
        sem = semantic_scores(query, candidates)
        # Movies not in semantic results get score 0
        scores = {
            mid: alpha * cf.get(mid, 0) + (1 - alpha) * sem.get(mid, 0)
            for mid in candidates
        }
    else:
        scores = cf

    top_ids = sorted(scores, key=lambda mid: -scores[mid])[:n]

    results = movies[movies["item_id"].isin(top_ids)][[
        "item_id", "title", "genres_text"
    ]].copy()
    results["score"] = results["item_id"].map(scores)
    results["cf_score"] = results["item_id"].map(cf)
    return results.sort_values("score", ascending=False).reset_index(drop=True)

## 5. Demo

In [8]:
# What has user 1 already watched?
user_id = 1
watched = ratings[ratings.user_id == user_id].merge(movies[["item_id", "title"]], on="item_id")
print(f"User {user_id} has rated {len(watched)} movies. Top-rated:")
watched.sort_values("rating", ascending=False)[["title", "rating"]].head(5)

User 1 has rated 272 movies. Top-rated:


,title,rating
136,Cinema Paradiso (1988),5
67,Brazil (1985),5
182,Maya Lin: A Strong Clear Vision (1994),5
186,Return of the Jedi (1983),5
191,Mystery Science Theater 3000: The Movie (1996),5


In [9]:
# Pure CF recommendations
print("=== Pure Collaborative Filtering (alpha=1.0) ===")
hybrid_recommend(user_id, query="", n=10, alpha=1.0)

=== Pure Collaborative Filtering (alpha=1.0) ===


,item_id,title,genres_text,score,cf_score
0,357,One Flew Over the Cuckoo's Nest (1975),Drama,1.000000,1.000000
1,474,Dr. Strangelove or: How I Learned to Stop Worr...,Sci-Fi War,1.000000,1.000000
2,603,Rear Window (1954),Mystery Thriller,1.000000,1.000000
3,513,"Third Man, The (1949)",Mystery Thriller,0.941002,0.941002
4,427,To Kill a Mockingbird (1962),Drama,0.939934,0.939934
5,302,L.A. Confidential (1997),Crime Film-Noir Mystery Thriller,0.921373,0.921373
6,483,Casablanca (1942),Drama Romance War,0.921122,0.921122
7,318,Schindler's List (1993),Drama War,0.915558,0.915558
8,408,"Close Shave, A (1995)",Animation Comedy Thriller,0.906306,0.906306
9,480,North by Northwest (1959),Comedy Thriller,0.899451,0.899451


In [10]:
# Hybrid: user history + semantic nudge
query = "psychological thriller with a dark twist"
print(f"=== Hybrid (alpha=0.6) | Query: '{query}' ===")
hybrid_recommend(user_id, query=query, n=10, alpha=0.6)

=== Hybrid (alpha=0.6) | Query: 'psychological thriller with a dark twist' ===


,item_id,title,genres_text,score,cf_score
0,302,L.A. Confidential (1997),Crime Film-Noir Mystery Thriller,0.847896,0.921373
1,480,North by Northwest (1959),Comedy Thriller,0.831294,0.899451
2,479,Vertigo (1958),Mystery Thriller,0.825840,0.886412
3,1143,Hard Eight (1996),Crime Thriller,0.805942,0.834424
4,1131,Safe (1995),Thriller,0.787643,0.814680
5,616,Night of the Living Dead (1968),Horror Sci-Fi,0.764643,0.780024
6,653,Touch of Evil (1958),Crime Film-Noir Thriller,0.744018,0.750338
7,607,Rebecca (1940),Romance Thriller,0.731501,0.740895
8,963,Some Folks Call It a Sling Blade (1993),Drama Thriller,0.729980,0.736454
9,1020,Gaslight (1944),Mystery Thriller,0.713665,0.681890


In [11]:
# Pure semantic — useful for cold-start (new user, no history)
query = "feel-good romantic comedy"
print(f"=== Pure Semantic (alpha=0.0) | Query: '{query}' ===")
hybrid_recommend(user_id, query=query, n=10, alpha=0.0)

=== Pure Semantic (alpha=0.0) | Query: 'feel-good romantic comedy' ===


,item_id,title,genres_text,score,cf_score
0,1190,That Old Feeling (1997),Comedy Romance,0.818843,0.441171
1,1351,Lover's Knot (1996),Comedy,0.784981,0.449927
2,1655,"Favor, The (1994)",Comedy Romance,0.784221,0.441573
3,1374,Falling in Love Again (1980),Comedy,0.783792,0.565871
4,1525,"Object of My Affection, The (1998)",Comedy Romance,0.781606,0.603618
5,1568,Vermont Is For Lovers (1992),Comedy Romance,0.781096,0.541769
6,1424,I Like It Like That (1994),Comedy Drama Romance,0.780681,0.480138
7,1611,Intimate Relations (1996),Comedy,0.779226,0.506322
8,1605,Love Serenade (1996),Comedy,0.777970,0.440623
9,1119,Some Kind of Wonderful (1987),Drama Romance,0.777569,0.557115


## 6. Evaluation

Beyond RMSE, we check **coverage** (how many unique items the system surfaces across all users) as a diversity metric.

In [12]:
from collections import Counter

# Sample 100 users and compare CF vs Hybrid recommendation overlap
sample_users = ratings["user_id"].drop_duplicates().sample(100, random_state=42).tolist()
query = "action adventure"

cf_recs, hybrid_recs = [], []
for uid in sample_users:
    cf_top = hybrid_recommend(uid, query="", n=10, alpha=1.0)["item_id"].tolist()
    hy_top = hybrid_recommend(uid, query=query, n=10, alpha=0.6)["item_id"].tolist()
    cf_recs.extend(cf_top)
    hybrid_recs.extend(hy_top)

cf_coverage    = len(set(cf_recs)) / len(all_movie_ids)
hybrid_coverage = len(set(hybrid_recs)) / len(all_movie_ids)

print(f"CF      coverage: {len(set(cf_recs))} unique movies ({cf_coverage:.1%} of catalog)")
print(f"Hybrid  coverage: {len(set(hybrid_recs))} unique movies ({hybrid_coverage:.1%} of catalog)")
print(f"Catalog size: {len(all_movie_ids)} movies")

CF      coverage: 202 unique movies (12.0% of catalog)
Hybrid  coverage: 76 unique movies (4.5% of catalog)
Catalog size: 1682 movies
